# PPLSTM vs xLSTM vs Transformer — simple GPU speed test

**Three models:**

1. **xLSTM (mLSTM + sLSTM)** — mLSTM first, then conventional sequential sLSTM.
2. **xLSTM (mLSTM + PPLSTM)** — the *same* mLSTM first, then our learned-permutation PPLSTM.
3. **Transformer** — two standard causal self-attention blocks.

**What is timed:** a complete *training* step: forward pass → cross-entropy → backward pass → AdamW update, using the **same random token inputs** at each shape. No dataset download or long training. The random tokens test **speed, not accuracy**.

**No custom speed-up kernels:** no Triton, FlashAttention, FlashRNN, CUDA Graphs, `torch.compile`, fused optimizers or `scaled_dot_product_attention`. The standard PyTorch operations still use their usual CUDA/cuBLAS kernels when a GPU is selected; these cannot be disabled without abandoning normal PyTorch.

The two xLSTM variants are **small, equations-based research implementations** based on our earlier PPLSTM experiments — **not the official NX-AI optimized xLSTM library**. The Transformer likewise uses *explicit* causal attention, not a fused attention kernel. Results therefore compare these plain-PyTorch implementations, **not maximum architecture performance**.

**Kaggle:** Settings → Accelerator → GPU; then **Run All**. One GPU is used even if Kaggle provides two T4s; this isolates single-GPU parallelism. The original uploaded notebook's multi-seed training/data/download/diagnostic sections are intentionally removed.


In [ ]:
import math, itertools, statistics, time, csv, platform
from pathlib import Path
import torch
from torch import nn
from torch.nn import functional as F

# Change these settings only. Smaller numbers = quicker benchmark.
WIDTH = 36             # must be divisible by 3 (PPLSTM) and by 4 (mLSTM heads)
VOCAB = 512            # synthetic token vocabulary; speed test only
HEADS = 4
BATCH_SEQUENCE = [(32, 32), (4, 64), (4, 128), (4, 256), (4, 512), (2, 1024)]
WARMUPS = 2
REPEATS = 4           # medians of complete training steps
SEED = 42
LR = 0.0003
RESULTS_CSV = 'plstm_xlstm_transformer_gpu_speed.csv'

assert WIDTH % 3 == 0 and WIDTH % HEADS == 0
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(SEED)
if DEVICE.type == 'cpu':
    torch.set_num_threads(min(2, torch.get_num_threads()))
print('Device:', DEVICE)
print('PyTorch:', torch.__version__)
print('CUDA device count:', torch.cuda.device_count())
if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('CPU ONLY: use Kaggle GPU accelerator for the intended comparison.')

# Turn off optional fast precision settings so all models have the same default math policy.
if DEVICE.type == 'cuda':
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False


## The three cells (ordinary PyTorch only)

The PPLSTM uses our **learned soft mixture of six 3×3 permutations** for each channel triplet. It keeps RWKV token mixing, signed carry and the stabilized sLSTM memory/normalizer (`c, n, m`).

Its update is `c_t = A_t c_(t-1) + b_t`, with `A_t = f_t P_t diag(sign_t)` and `b_t = i_t z_t`. The 3×3 affine updates are combined via an associative prefix scan. The standard sLSTM baseline instead calculates gates from its **actual previous hidden state** step by step.


In [ ]:
def shift_right(x):
    return torch.cat((torch.zeros_like(x[:, :1]), x[:, :-1]), dim=1)


def affine_scan(a, b):
    # Exact parallel-prefix algebra for per-element affine updates: x'=a*x+b.
    # This is unoptimized ordinary PyTorch; neither a CUDA graph nor a custom kernel.
    A, B = a, b
    stride = 1
    while stride < a.shape[1]:
        A, B = (torch.cat((A[:, :stride], A[:, stride:] * A[:, :-stride]), dim=1),
                torch.cat((B[:, :stride], B[:, stride:] + A[:, stride:] * B[:, :-stride]), dim=1))
        stride *= 2
    return A, B


def matrix_scan(A, b):
    # A: [batch,time,groups,3,3], b: [batch,time,groups,3].
    stride = 1
    while stride < A.shape[1]:
        new_A = torch.cat((A[:, :stride], A[:, stride:] @ A[:, :-stride]), dim=1)
        new_b = torch.cat((b[:, :stride], b[:, stride:] +
                           (A[:, stride:] @ b[:, :-stride].unsqueeze(-1)).squeeze(-1)), dim=1)
        A, b = new_A, new_b
        stride *= 2
    return A, b


class SequentialSLSTM(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.xproj = nn.Linear(width, 4 * width)
        self.hproj = nn.Linear(width, 4 * width, bias=False)
        nn.init.orthogonal_(self.hproj.weight)
        self.input_bias = nn.Parameter(torch.full((width,), -2.0))
        self.forget_bias = nn.Parameter(torch.full((width,), 2.0))

    def forward(self, x):
        batch, length, width = x.shape
        gates = self.xproj(x)
        h = c = n = m = x.new_zeros(batch, width)
        outputs = []
        for t in range(length):
            gi, gf, gz, go = (gates[:, t] + self.hproj(h)).chunk(4, dim=-1)
            li = gi + self.input_bias
            lf = F.logsigmoid(gf + self.forget_bias)
            m_next = torch.maximum(li, m + lf)
            f = (lf + m - m_next).exp()
            i = (li - m_next).exp()
            c = f * c + i * gz.tanh()
            n = f * n + i
            h = go.sigmoid() * (c / n.clamp_min(1e-12))
            m = m_next
            outputs.append(h)
        return torch.stack(outputs, dim=1)


class PermutationPLSTM(nn.Module):
    """Matches the earlier learned 3-channel routing RWKV/PPLSTM research cell."""
    def __init__(self, width):
        super().__init__()
        assert width % 3 == 0
        self.width, self.groups = width, width // 3
        self.xproj = nn.Linear(width, 4 * width)
        self.mix_logit = nn.Parameter(torch.full((4, width), math.log(3.0)))
        self.sign_proj = nn.Linear(width, width)
        self.initial_cell = nn.Parameter(torch.linspace(-1.5, 1.5, 3).repeat(self.groups))
        self.input_bias = nn.Parameter(torch.full((width,), -4.0))
        self.forget_bias = nn.Parameter(torch.full((width,), 4.0))
        nn.init.zeros_(self.sign_proj.weight)
        nn.init.constant_(self.sign_proj.bias, 2.8)
        bank = []
        for order in itertools.permutations(range(3)):
            mat = torch.zeros(3, 3)
            for row, col in enumerate(order):
                mat[row, col] = 1.0
            bank.append(mat)
        self.register_buffer('perm_bank', torch.stack(bank))  # [6,3,3]
        self.selector = nn.Linear(width, self.groups * 6)
        nn.init.zeros_(self.selector.weight)
        nn.init.constant_(self.selector.bias, -1.0)
        with torch.no_grad():
            for group in range(self.groups):
                self.selector.bias[group * 6] = 3.0

    def forward(self, x):
        batch, length, width = x.shape
        groups = self.groups
        previous_input = shift_right(x)
        mix_coeff = self.mix_logit.sigmoid().view(1, 1, 4, width)
        gate_values = (self.xproj(x).reshape(batch, length, 4, width) * mix_coeff
                       + self.xproj(previous_input).reshape(batch, length, 4, width) * (1 - mix_coeff))
        li, lf, gz, go = gate_values.unbind(dim=2)
        mix = x * self.mix_logit[0].sigmoid() + previous_input * (1-self.mix_logit[0].sigmoid())
        signs = self.sign_proj(mix).tanh().reshape(batch, length, groups, 3)
        z = gz.tanh().reshape(batch, length, groups, 3)
        logi = (li.reshape(batch, length, groups, 3) + self.input_bias.reshape(groups, 3)).mean(-1)
        logf = F.logsigmoid((lf.reshape(batch, length, groups, 3) +
                               self.forget_bias.reshape(groups, 3)).mean(-1))
        lf_prefix = torch.cumsum(logf, dim=1)
        m = lf_prefix + torch.cummax(torch.cat((torch.zeros_like(lf_prefix[:, :1]),
                         logi - lf_prefix), dim=1), dim=1).values[:, 1:]
        forget = (logf + shift_right(m) - m).exp()
        insert = (logi - m).exp()
        weights = self.selector(mix).reshape(batch, length, groups, 6).softmax(-1)
        P = torch.einsum('btgp,pij->btgij', weights, self.perm_bank)
        A = forget[..., None, None] * P * signs[..., None, :]
        b = insert[..., None] * z
        prefix_A, prefix_b = matrix_scan(A, b)
        c0 = self.initial_cell.tanh().reshape(1, groups, 3).expand(batch, -1, -1)
        c = (prefix_A @ c0[:, None, :, :, None]).squeeze(-1) + prefix_b
        norm_a, norm_b = affine_scan(forget, insert)
        n = norm_a + norm_b   # initial normalizer is 1
        ratio = (c / n[..., None].clamp_min(1e-12)).reshape(batch, length, width)
        return go.sigmoid() * ratio


class MLSTM(nn.Module):
    def __init__(self, width, heads=4):
        super().__init__()
        assert width % heads == 0
        self.heads, self.dim = heads, width // heads
        self.qkv = nn.Linear(width, 3 * width)
        self.gate = nn.Linear(width, 2 * heads + width)
        self.input_bias = nn.Parameter(torch.full((heads,), -2.0))
        self.forget_bias = nn.Parameter(torch.full((heads,), 2.0))

    def forward(self, x):
        batch, length, width = x.shape
        heads, d = self.heads, self.dim
        q, k, v = self.qkv(x).reshape(batch, length, 3, heads, d).unbind(2)
        k = k / math.sqrt(d)
        li, lf, go = torch.split(self.gate(x), [heads, heads, width], dim=-1)
        li = li + self.input_bias
        lf = F.logsigmoid(lf + self.forget_bias)
        cumulative = torch.cumsum(lf, dim=1)
        m = cumulative + torch.cummax(torch.cat((torch.zeros_like(cumulative[:, :1]),
                        li - cumulative), dim=1), dim=1).values[:, 1:]
        forget = (lf + shift_right(m) - m).exp()
        insert = (li - m).exp()
        memory_write = insert[..., None, None] * v[..., :, None] * k[..., None, :]
        C = affine_scan(forget[..., None, None], memory_write)[1]
        N = affine_scan(forget[..., None], insert[..., None] * k)[1]
        numerator = (C @ q.unsqueeze(-1)).squeeze(-1)
        denominator = torch.maximum((N * q).sum(-1).abs(), (-m).exp()).clamp_min(1e-8)
        return go.sigmoid() * (numerator / denominator[..., None]).reshape(batch, length, width)


class ExplicitCausalAttention(nn.Module):
    """No scaled_dot_product_attention, FlashAttention, or KV cache."""
    def __init__(self, width, heads=4):
        super().__init__()
        assert width % heads == 0
        self.heads, self.dim = heads, width // heads
        self.qkv = nn.Linear(width, 3 * width)
        self.out = nn.Linear(width, width)

    def forward(self, x):
        batch, length, width = x.shape
        h, d = self.heads, self.dim
        q, k, v = self.qkv(x).reshape(batch, length, 3, h, d).unbind(2)
        q, k, v = (y.transpose(1, 2) for y in (q, k, v))
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(d)
        causal = torch.ones(length, length, dtype=torch.bool, device=x.device).tril()
        probs = torch.softmax(scores.masked_fill(~causal, -1e9), dim=-1)
        context = (probs @ v).transpose(1, 2).reshape(batch, length, width)
        return self.out(context)


class SpeedModel(nn.Module):
    """Two residual layers; same embedding/FF/output across all three families."""
    def __init__(self, name, vocab=512, width=36, heads=4):
        super().__init__()
        self.name = name
        self.embedding = nn.Embedding(vocab, width)
        self.input_norms = nn.ModuleList([nn.LayerNorm(width) for _ in range(2)])
        self.cells = nn.ModuleList(
            [MLSTM(width, heads), SequentialSLSTM(width)] if name == 'xLSTM (mLSTM + sLSTM)' else
            [MLSTM(width, heads), PermutationPLSTM(width)] if name == 'xLSTM (mLSTM + PPLSTM)' else
            [ExplicitCausalAttention(width, heads), ExplicitCausalAttention(width)]
        )
        self.projections = nn.ModuleList([nn.Linear(width, width) for _ in range(2)])
        self.ff_norms = nn.ModuleList([nn.LayerNorm(width) for _ in range(2)])
        self.ff = nn.ModuleList([nn.Sequential(nn.Linear(width, 2 * width), nn.GELU(),
                                             nn.Linear(2 * width, width)) for _ in range(2)])
        self.output_norm = nn.LayerNorm(width)
        self.lm_head = nn.Linear(width, vocab, bias=False)
        self.lm_head.weight = self.embedding.weight

    def forward(self, tokens):
        width = self.embedding.embedding_dim
        x = self.embedding(tokens) * math.sqrt(width)
        for norm, cell, proj, f_norm, ff in zip(self.input_norms, self.cells, self.projections,
                                                 self.ff_norms, self.ff):
            x = x + proj(cell(norm(x)))
            x = x + ff(f_norm(x))
        return self.lm_head(self.output_norm(x)) / math.sqrt(width)


## Quick sanity checks

Before timing, this checks that every model produces finite loss/gradients, the PPLSTM associative scan matches a short serial recurrence using the **same learned transformations**, and no model sees future tokens. No quality training is performed.


In [ ]:
def make_model(name, width=WIDTH):
    torch.manual_seed(SEED)    # repeatable initialization for each model
    return SpeedModel(name, VOCAB, width, HEADS).to(DEVICE)

NAMES = ['xLSTM (mLSTM + sLSTM)', 'xLSTM (mLSTM + PPLSTM)', 'Transformer']
for name in NAMES:
    model = make_model(name).eval()
    sample = torch.randint(0, VOCAB, (2, 12), device=DEVICE)
    altered = sample.clone()
    altered[:, 6:] = (altered[:, 6:] + 1) % VOCAB
    with torch.no_grad():
        z = model(sample)
        z_altered = model(altered)
        causal_error = (z[:, :6] - z_altered[:, :6]).abs().max().item()
    assert z.shape == (2, 12, VOCAB) and torch.isfinite(z).all()
    assert causal_error < 2e-4, (name, causal_error)
    params = sum(p.numel() for p in model.parameters())
    print(f'{name}: {params:,} parameters | causality error: {causal_error:.2g}')
    del model

# Verify the underlying PPLSTM prefix operation against serial evaluation.
torch.manual_seed(7)
A = torch.randn(2, 8, 3, 3, 3, device=DEVICE) * 0.12
b = torch.randn(2, 8, 3, 3, device=DEVICE) * 0.12
_, prefixes = matrix_scan(A, b)
serial = torch.zeros_like(b[:, 0])
serial_list = []
for t in range(b.shape[1]):
    serial = (A[:, t] @ serial[..., None]).squeeze(-1) + b[:, t]
    serial_list.append(serial)
scan_error = (prefixes - torch.stack(serial_list, 1)).abs().max().item()
print('PPLSTM affine prefix vs serial max difference:', scan_error)
assert scan_error < 2e-4
print('Sanity checks passed.')


## Benchmark — run this cell

**All three models are untrained.** Their accuracy on random tokens is irrelevant. Times include forward + backward + **unfused** AdamW and exclude model construction and random-token generation. CUDA synchronization is included so GPU timings are real elapsed step times, not asynchronous launch times.

By default the notebook tests 32, 64, 128, 256, 512 and 1,024 token lengths. On a slow GPU, remove the longer rows in `BATCH_SEQUENCE` or reduce `REPEATS` in the first settings cell. The models run **one after another on the same GPU** — not one model per GPU.


In [ ]:
def sync():
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize(DEVICE)


def training_step(model, optimizer, input_ids, targets):
    optimizer.zero_grad(set_to_none=True)
    logits = model(input_ids)
    loss = F.cross_entropy(logits.reshape(-1, VOCAB), targets.reshape(-1))
    loss.backward()
    optimizer.step()
    return loss


rows = []
for batch, length in BATCH_SEQUENCE:
    # Identical synthetic tokens for all three models at this shape.
    torch.manual_seed(SEED + batch + length)
    x = torch.randint(0, VOCAB, (batch, length), device=DEVICE)
    y = torch.randint(0, VOCAB, (batch, length), device=DEVICE)
    for name in NAMES:
        model = None
        optimizer = None
        try:
            model = make_model(name).train()
            # Disable fused/foreach optimizer implementations explicitly.
            optimizer = torch.optim.AdamW(model.parameters(), lr=LR, fused=False, foreach=False)
            for _ in range(WARMUPS):
                training_step(model, optimizer, x, y)
            sync()
            if DEVICE.type == 'cuda':
                torch.cuda.reset_peak_memory_stats(DEVICE)
            samples_ms = []
            for _ in range(REPEATS):
                sync()
                start = time.perf_counter()
                loss = training_step(model, optimizer, x, y)
                sync()
                samples_ms.append(1000 * (time.perf_counter() - start))
            median_ms = statistics.median(samples_ms)
            row = {'model': name, 'batch': batch, 'seq_len': length,
                   'tokens_per_step': batch * length,
                   'parameters': sum(p.numel() for p in model.parameters()),
                   'median_step_ms': round(median_ms, 2),
                   'tokens_per_sec': round(batch * length * 1000 / median_ms, 1),
                   'peak_allocated_MiB': round(torch.cuda.max_memory_allocated(DEVICE)/1024**2, 1)
                                      if DEVICE.type == 'cuda' else None,
                   'device': str(DEVICE), 'repeats': REPEATS, 'status': 'OK'}
        except (RuntimeError, MemoryError) as ex:
            sync()
            row = {'model': name, 'batch': batch, 'seq_len': length,
                   'tokens_per_step': batch * length, 'parameters': None,
                   'median_step_ms': None, 'tokens_per_sec': None,
                   'peak_allocated_MiB': None, 'device': str(DEVICE),
                   'repeats': REPEATS, 'status': f'FAILED: {str(ex)[:110]}'}
            if DEVICE.type == 'cuda':
                torch.cuda.empty_cache()
        rows.append(row)
        print(f"B={batch:>2} T={length:>4} | {name:25} | "
              f"{str(row['median_step_ms']):>8} ms | {str(row['tokens_per_sec']):>9} tok/s | {row['status']}", flush=True)
        del model, optimizer
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()  # between models only, never inside a timed step

import pandas as pd
from IPython.display import display
results = pd.DataFrame(rows)
results.to_csv(RESULTS_CSV, index=False)
print('\nSaved:', Path(RESULTS_CSV).resolve())
display(results[['model', 'batch', 'seq_len', 'median_step_ms', 'tokens_per_sec',
                 'peak_allocated_MiB', 'parameters', 'status']])

# Speedup > 1 means PPLSTM is faster than sequential sLSTM *in this implementation*.
pivot = results.pivot_table(index=['batch','seq_len'], columns='model',
                            values='median_step_ms', aggfunc='first')
a, b = 'xLSTM (mLSTM + sLSTM)', 'xLSTM (mLSTM + PPLSTM)'
if a in pivot.columns and b in pivot.columns:
    pivot['PPLSTM speedup over sLSTM'] = (pivot[a] / pivot[b]).round(2)
    display(pivot)


### Reading the results

- **`median_step_ms`**: full training-step time, lower is better.
- **`tokens_per_sec`**: tokens processed per second, higher is better. These are randomly generated token IDs, not learned language quality.
- **`PPLSTM speedup over sLSTM`**: greater than `1.0` means our PPLSTM hybrid is faster **for that exact batch/sequence combination**.
- **`peak_allocated_MiB`**: approximate GPU memory used by the model, activations and optimizer; no cross-GPU memory summation.

**Important:** a plain PyTorch permutation scan creates many temporary matrices and GPU launches. It may be **slower** than sLSTM here despite being time-parallel mathematically. That is still a valid finding and helps decide whether future kernel work is worthwhile. This notebook does **not** benchmark the production xLSTM CUDA kernels or distributed training; it is a deliberately simple, reproducible one-GPU microbenchmark.
